# 直近得点者モデル（Recent Scorer Model）：Dixon & Robinson (1998) の試合状況分類の改変

`dixon_robinson_model.ipynb`（純粋なモデルVI）をベースに、**スコア状態による倍率の分類だけ**を
「リード／同点／ビハインド × 直前に点を取ったのは自チームか相手か」に置き換えたモデル。
それ以外（攻撃力 α・守備力 β、ホームアドバンテージ γ_h、アディショナルタイム補正 ρ1, ρ2、
線形トレンド ξ1, ξ2、退場者効果なし）は元のモデルと同じ。

### 試合状況の分類（各チームから見た7区分）

| 記号 | 状態 |
|---|---|
| `00` | 0-0（基準状態、倍率 = 1） |
| `lead_scored` | リード中かつ直近得点（最後のゴールが自チーム） |
| `lead_conceded` | リード中かつ直近失点（最後のゴールが相手） |
| `behind_scored` | ビハインド中かつ直近得点 |
| `behind_conceded` | ビハインド中かつ直近失点 |
| `level_scored` | 同点（0-0以外）かつ直近得点 |
| `level_conceded` | 同点（0-0以外）かつ直近失点 |

「直近」は時間窓を設けず、**最後に得点したのがどちらのチームか**だけで決める
（状態が変わるのはゴールの瞬間のみ）。

### 2つのモデル

- **共有モデル（shared）**：ホームとアウェイで同じ倍率 θ_s を使う（状態パラメータ6個）
- **分離モデル（separate）**：ホーム用 θ^H_s とアウェイ用 θ^A_s を別々に推定（状態パラメータ12個）

共有モデルは分離モデルの制約付き版（θ^H = θ^A）なので、尤度比検定（自由度6）と AIC/BIC で比較する。

- 入力：`Goals_and_red_cards.csv` を `pandas` で読み込んだ DataFrame（1リーグ・1シーズン分）
- 出力：各モデルの最尤推定値、対数尤度、AIC、BIC、および両モデルの比較

## 想定するデータ形式

元のノートブックと同じ（`match_id`, `home_team`, `away_team`, `home_away`, `red_card`, `dr_time`）。
退場者の行は無視し、0-0 の試合のプレースホルダー行（`dr_time` 欠損）は「イベント無しの試合」として扱う。

In [1]:
import glob
import os

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2

/Users/fujitaharukanin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## モデルの定式化

試合 $k$（ホーム $i$、アウェイ $j$）で、時刻 $t$ における両チームの状態を
$s_H(t), s_A(t) \in \{00, \text{lead\_scored}, \dots, \text{level\_conceded}\}$ とする。
両者は1対1に対応している（例：ホームが「リード中かつ直近得点」ならアウェイは「ビハインド中かつ直近失点」）。

$$\lambda_k^*(t) = \rho(t)\Big(\alpha_i\beta_j\gamma_h \cdot \theta^{H}_{s_H(t)} + \xi_1 t\Big)$$
$$\mu_k^*(t) = \rho(t)\Big(\alpha_j\beta_i \cdot \theta^{A}_{s_A(t)} + \xi_2 t\Big)$$

- 共有モデル：$\theta^H_s = \theta^A_s = \theta_s$
- 分離モデル：$\theta^H_s, \theta^A_s$ を別々に推定
- どちらも $\theta_{00} = 1$（基準）

**λ と μ（ホーム・アウェイの2本の強度）は残る**点に注意。どちらのチームが点を取るかの
2通りがある以上、強度は2本必要。今回の分類でまとめられる可能性があるのは、
状態倍率の「表」（元モデルの `lambda_xy` と `mu_xy`）のほう。

尤度・識別性制約（$\text{mean}(\alpha)=1$）は元のモデルと同じ。

In [2]:
# インジュリータイム（アディショナルタイム）とみなす区間の境界（元のモデルと同じ）
INJ1_START, INJ1_END = 44 / 90, 45 / 90
INJ2_START, INJ2_END = 89 / 90, 90 / 90

# 基準状態 "00" 以外の6状態（この順でパラメータを並べる）
STATES = [
    "lead_scored", "lead_conceded",
    "behind_scored", "behind_conceded",
    "level_scored", "level_conceded",
]
N_STATES = len(STATES)


class _InvalidRate(Exception):
    # 得点強度が非正になってしまった場合に投げる内部例外（最適化の枝刈り用）
    pass


def _team_states(x, y, last):
    '''
    スコア (x, y) と最後に得点した側 last ("H" / "A" / None) から、
    (ホームから見た状態, アウェイから見た状態) を返す。
    '''
    if last is None:              # まだ誰も得点していない = 0-0
        return "00", "00"
    diff = x - y
    if diff > 0:
        h_rel, a_rel = "lead", "behind"
    elif diff < 0:
        h_rel, a_rel = "behind", "lead"
    else:
        h_rel, a_rel = "level", "level"
    if last == "H":
        return f"{h_rel}_scored", f"{a_rel}_conceded"
    return f"{h_rel}_conceded", f"{a_rel}_scored"

## データの前処理

元のノートブックと同じ。

In [3]:
def prepare_matches(df):
    '''
    Goals_and_red_cards の df を、試合ごとのイベント列に変換する（退場者イベントは無視）。
    戻り値: list of {"home", "away", "events": [(t, "H_GOAL"/"A_GOAL"), ...]}
    '''
    matches = {}
    d = df.sort_values(["match_id", "dr_time"])
    for row in d.itertuples(index=False):
        mid = row.match_id
        if mid not in matches:
            matches[mid] = {"home": row.home_team, "away": row.away_team, "events": []}
        if pd.isna(row.dr_time):
            continue      # 0-0 の試合のプレースホルダー行
        if pd.notna(row.red_card):
            continue      # 退場者イベントは使わない
        t = float(row.dr_time)
        kind = "H_GOAL" if int(row.home_away) == 0 else "A_GOAL"
        matches[mid]["events"].append((t, kind))
    return list(matches.values())


def build_index(matches):
    teams = sorted(set([m["home"] for m in matches] + [m["away"] for m in matches]))
    idx = {t: i for i, t in enumerate(teams)}
    return teams, idx

## 状態ごとのサンプル数の確認

「リード中かつ直近失点」などは合計2点以上ないと発生しないため、状態によってはデータが薄い。
推定の前に、各状態での **滞在時間（試合数換算）** と **得点数** を数えておく。

In [4]:
def state_occupancy(matches):
    '''
    各状態について、その状態にいたチームの延べ滞在時間（1 = 1試合分）と、
    その状態のチームが挙げた得点数を集計する（ホーム・アウェイ別）。
    '''
    rows = {}
    def add(state, side, dt, goal):
        r = rows.setdefault((state, side), {"state": state, "side": side, "exposure": 0.0, "goals": 0})
        r["exposure"] += dt
        r["goals"] += goal

    for m in matches:
        x = y = 0
        last = None
        t_prev = 0.0
        for t_ev, kind in sorted(m["events"], key=lambda e: e[0]):
            sh, sa = _team_states(x, y, last)
            add(sh, "home", t_ev - t_prev, int(kind == "H_GOAL"))
            add(sa, "away", t_ev - t_prev, int(kind == "A_GOAL"))
            if kind == "H_GOAL":
                x += 1; last = "H"
            else:
                y += 1; last = "A"
            t_prev = t_ev
        sh, sa = _team_states(x, y, last)
        add(sh, "home", 1.0 - t_prev, 0)
        add(sa, "away", 1.0 - t_prev, 0)

    out = pd.DataFrame(rows.values())
    out["goals_per_match"] = out["goals"] / out["exposure"].where(out["exposure"] > 0)
    order = {s: i for i, s in enumerate(["00"] + STATES)}
    out = out.sort_values(["side", "state"], key=lambda c: c.map(order) if c.name == "state" else c)
    return out.reset_index(drop=True)

## パラメータと尤度関数

共通パラメータベクトル `g` の並びは

- 共有モデル：`gamma_h, rho1, rho2, theta_<6状態>, xi1, xi2`（11個）
- 分離モデル：`gamma_h, rho1, rho2, thetaH_<6状態>, thetaA_<6状態>, xi1, xi2`（17個）

正値制約が必要なもの（α, β, γ_h, ρ, θ）は `exp()` で正値化し、ξ は実数のまま最適化する。

In [5]:
def global_param_names(shared):
    if shared:
        th = [f"theta_{s}" for s in STATES]
    else:
        th = [f"thetaH_{s}" for s in STATES] + [f"thetaA_{s}" for s in STATES]
    return ["gamma_h", "rho1", "rho2"] + th + ["xi1", "xi2"]


def unpack_params(theta, n_teams, shared):
    a = theta[0:n_teams]
    b = theta[n_teams:2 * n_teams]
    g = theta[2 * n_teams:]

    alpha = np.exp(a)
    alpha = alpha / alpha.mean()      # 制約 mean(alpha) = 1
    beta = np.exp(b)

    gamma_h, rho1, rho2 = np.exp(g[0]), np.exp(g[1]), np.exp(g[2])
    th_h = {"00": 1.0}
    th_a = {"00": 1.0}
    for k, s in enumerate(STATES):
        th_h[s] = np.exp(g[3 + k])
        th_a[s] = np.exp(g[3 + k]) if shared else np.exp(g[3 + N_STATES + k])
    xi1, xi2 = g[-2], g[-1]
    return alpha, beta, gamma_h, rho1, rho2, th_h, th_a, xi1, xi2

In [6]:
def _check_pos(base, xi, t):
    if base + xi * t <= 0:
        raise _InvalidRate()


def _rho_at(t, rho1, rho2):
    if INJ1_START < t <= INJ1_END:
        return rho1
    if INJ2_START < t <= INJ2_END:
        return rho2
    return 1.0


def _integrate_rate(t1, t2, base, xi, rho1, rho2):
    # 区間 [t1, t2]（状態一定）で ∫ rho(t)*(base + xi*t) dt
    bpoints = sorted({t1, t2} | {p for p in (INJ1_START, INJ1_END, INJ2_START, INJ2_END) if t1 < p < t2})
    for p in bpoints:
        _check_pos(base, xi, p)
    total = 0.0
    for a, c in zip(bpoints[:-1], bpoints[1:]):
        r = _rho_at(0.5 * (a + c), rho1, rho2)
        total += r * (base * (c - a) + xi * (c ** 2 - a ** 2) / 2.0)
    return total


def _rate_at(t, base, xi, rho1, rho2):
    val = base + xi * t
    if val <= 0:
        raise _InvalidRate()
    return _rho_at(t, rho1, rho2) * val


def _match_loglik(match, idx, alpha, beta, gamma_h, rho1, rho2, th_h, th_a, xi1, xi2):
    hi, aj = idx[match["home"]], idx[match["away"]]
    lam_k = alpha[hi] * beta[aj] * gamma_h   # home側の基礎強度（0-0時）
    mu_k = alpha[aj] * beta[hi]              # away側の基礎強度（0-0時）

    x = y = 0
    last = None
    t_prev = 0.0
    ll = 0.0
    for t_ev, kind in sorted(match["events"], key=lambda e: e[0]):
        sh, sa = _team_states(x, y, last)
        base_h = lam_k * th_h[sh]
        base_a = mu_k * th_a[sa]
        ll -= _integrate_rate(t_prev, t_ev, base_h, xi1, rho1, rho2)
        ll -= _integrate_rate(t_prev, t_ev, base_a, xi2, rho1, rho2)
        if kind == "H_GOAL":
            ll += np.log(_rate_at(t_ev, base_h, xi1, rho1, rho2))
            x += 1; last = "H"
        else:
            ll += np.log(_rate_at(t_ev, base_a, xi2, rho1, rho2))
            y += 1; last = "A"
        t_prev = t_ev

    sh, sa = _team_states(x, y, last)
    ll -= _integrate_rate(t_prev, 1.0, lam_k * th_h[sh], xi1, rho1, rho2)
    ll -= _integrate_rate(t_prev, 1.0, mu_k * th_a[sa], xi2, rho1, rho2)
    return ll


def negative_log_likelihood(theta, matches, idx, shared):
    params = unpack_params(theta, len(idx), shared)
    total = 0.0
    for m in matches:
        try:
            total += _match_loglik(m, idx, *params)
        except _InvalidRate:
            total += -1e9   # 強度が負になる領域には大きなペナルティ
    if not np.isfinite(total):
        return 1e12
    return -total


def _count_invalid_matches(theta, matches, idx, shared):
    params = unpack_params(theta, len(idx), shared)
    n_invalid = 0
    for m in matches:
        try:
            _match_loglik(m, idx, *params)
        except _InvalidRate:
            n_invalid += 1
    return n_invalid

## 初期値の設定

チーム別の初期値は元のノートブックと同じ作り方。状態倍率は1（log = 0）から始める。

In [7]:
def initial_theta(matches, idx, shared):
    n_teams = len(idx)
    goals_for = np.zeros(n_teams)
    goals_against = np.zeros(n_teams)
    games = np.zeros(n_teams)
    total_goals = total_games = home_goals_total = away_goals_total = 0
    for m in matches:
        hi, aj = idx[m["home"]], idx[m["away"]]
        xg = sum(1 for t, k in m["events"] if k == "H_GOAL")
        yg = sum(1 for t, k in m["events"] if k == "A_GOAL")
        goals_for[hi] += xg; goals_against[aj] += xg
        goals_for[aj] += yg; goals_against[hi] += yg
        games[hi] += 1; games[aj] += 1
        total_goals += xg + yg; total_games += 1
        home_goals_total += xg; away_goals_total += yg

    avg = total_goals / max(2 * total_games, 1)
    att0 = np.clip(np.where(games > 0, (goals_for / np.maximum(games, 1)) / avg, 1.0), 0.3, 3.0)
    def0 = np.clip(np.where(games > 0, (goals_against / np.maximum(games, 1)) / avg, 1.0), 0.3, 3.0)

    g0 = np.zeros(len(global_param_names(shared)))
    g0[0] = np.log(max(home_goals_total / max(away_goals_total, 1), 0.1))
    return np.concatenate([np.log(att0), np.log(def0), g0])


def make_bounds(n_teams, shared):
    n_theta = N_STATES if shared else 2 * N_STATES
    bounds = [(-3, 3)] * (2 * n_teams)
    bounds += [(-3, 3)] * 3          # gamma_h, rho1, rho2
    bounds += [(-3, 3)] * n_theta    # 状態倍率（logスケール）
    bounds += [(-5, 5), (-5, 5)]     # xi1, xi2
    return bounds


def shared_to_separate(theta_shared, n_teams):
    # 共有モデルの推定値を分離モデルの初期値に変換する（thetaH = thetaA = theta）
    teams_part = theta_shared[:2 * n_teams]
    g = theta_shared[2 * n_teams:]
    th = g[3:3 + N_STATES]
    return np.concatenate([teams_part, g[:3], th, th, g[-2:]])

## 推定（最適化）

元のノートブックと同じく L-BFGS-B → Powell の2段階。

分離モデルは **共有モデルの推定値を初期値にする**（`theta_init` 引数）。
分離モデルは共有モデルを含むので、こうしておけば分離モデルの対数尤度が共有モデルを
下回る（＝最適化の失敗で尤度比検定が壊れる）ことを防げる。

In [8]:
def fit_recent_scorer(df, shared=True, theta_init=None, maxiter=1000, disp=False, verbose=True):
    '''
    直近得点者モデルを推定する。

    Parameters
    ----------
    df : pandas.DataFrame   1リーグ・1シーズン分の Goals_and_red_cards
    shared : bool           True なら共有モデル、False なら分離モデル
    theta_init : ndarray    初期値（None なら initial_theta）
    '''
    matches = prepare_matches(df)
    teams, idx = build_index(matches)
    n_teams = len(teams)

    theta0 = initial_theta(matches, idx, shared) if theta_init is None else np.asarray(theta_init, float)
    bounds = make_bounds(n_teams, shared)

    res1 = minimize(
        negative_log_likelihood, theta0, args=(matches, idx, shared),
        method="L-BFGS-B", bounds=bounds,
        options={"maxiter": maxiter, "maxfun": maxiter * 50},
    )
    # L-BFGS-B がペナルティ領域に飛び込んで初期値より悪化した場合は、初期値から Powell を始める
    f0 = negative_log_likelihood(theta0, matches, idx, shared)
    start2 = res1.x if res1.fun <= f0 else theta0
    res = minimize(
        negative_log_likelihood, start2, args=(matches, idx, shared),
        method="Powell", bounds=bounds,
        options={"maxiter": maxiter * 20, "maxfev": maxiter * 200, "xtol": 1e-10, "ftol": 1e-12},
    )
    if disp:
        print(f"[stage1: L-BFGS-B] loglik={-res1.fun:.4f} success={res1.success}")
        print(f"[stage2: Powell]   loglik={-res.fun:.4f} success={res.success}")

    theta = res.x
    alpha, beta, gamma_h, rho1, rho2, th_h, th_a, xi1, xi2 = unpack_params(theta, n_teams, shared)

    n_params = len(theta)
    loglik = -res.fun
    aic = 2 * n_params - 2 * loglik
    bic = n_params * np.log(len(matches)) - 2 * loglik   # n = 試合数

    team_params = pd.DataFrame({"team": teams, "alpha_attack": alpha, "beta_defence": beta})

    names = global_param_names(shared)
    g = theta[2 * n_teams:]
    est = [np.exp(v) for v in g[:-2]] + [g[-2], g[-1]]
    global_params = pd.DataFrame({"parameter": names, "estimate": est})

    # 状態倍率を見やすい表にしたもの
    state_table = pd.DataFrame({
        "state": ["00"] + STATES,
        "theta_home": [th_h[s] for s in ["00"] + STATES],
        "theta_away": [th_a[s] for s in ["00"] + STATES],
    })

    n_invalid = _count_invalid_matches(theta, matches, idx, shared)
    message = str(res.message)
    if n_invalid > 0:
        message = (f"[WARNING] {n_invalid} match(es) still have non-positive scoring intensity; "
                   f"log-likelihood/AIC/BIC are not reliable. " + message)

    summary = {
        "model": "shared" if shared else "separate",
        "n_matches": len(matches), "n_teams": n_teams, "n_params": n_params,
        "log_likelihood": loglik, "AIC": aic, "BIC": bic,
        "converged": bool(res.success), "message": message,
    }

    if verbose:
        print(f"==== 直近得点者モデル（{summary['model']}）====")
        print(f"試合数: {summary['n_matches']}, チーム数: {n_teams}, パラメータ数: {n_params}")
        print(f"対数尤度: {loglik:.3f}  AIC: {aic:.3f}  BIC: {bic:.3f}")
        print(f"収束: {summary['converged']} ({message})")
        print()
        print("---- 状態倍率 ----")
        print(state_table.to_string(index=False))
        print()
        print("---- 共通パラメータ ----")
        print(global_params.to_string(index=False))

    return summary, team_params, global_params, state_table, res


def compare_models(summary_shared, summary_separate):
    '''共有モデル（帰無仮説 thetaH = thetaA）と分離モデルの尤度比検定'''
    lr = 2 * (summary_separate["log_likelihood"] - summary_shared["log_likelihood"])
    dof = summary_separate["n_params"] - summary_shared["n_params"]
    p = chi2.sf(max(lr, 0.0), dof)
    return {
        "LR_stat": lr, "df": dof, "p_value": p,
        "AIC_shared": summary_shared["AIC"], "AIC_separate": summary_separate["AIC"],
        "BIC_shared": summary_shared["BIC"], "BIC_separate": summary_separate["BIC"],
    }


def fit_both(df, disp=False, verbose=True):
    '''共有モデル → （その推定値を初期値に）分離モデル の順に推定し、比較結果も返す'''
    s_sh, tp_sh, gp_sh, st_sh, res_sh = fit_recent_scorer(df, shared=True, disp=disp, verbose=verbose)
    if verbose:
        print()
    n_teams = s_sh["n_teams"]
    s_sep, tp_sep, gp_sep, st_sep, res_sep = fit_recent_scorer(
        df, shared=False, theta_init=shared_to_separate(res_sh.x, n_teams), disp=disp, verbose=verbose)
    cmp = compare_models(s_sh, s_sep)
    if verbose:
        print()
        print("==== 共有 vs 分離 ====")
        print(f"LR = {cmp['LR_stat']:.3f} (df = {cmp['df']}), p = {cmp['p_value']:.4f}")
        print(f"AIC: shared {cmp['AIC_shared']:.3f} / separate {cmp['AIC_separate']:.3f}")
        print(f"BIC: shared {cmp['BIC_shared']:.3f} / separate {cmp['BIC_separate']:.3f}")
    return {
        "shared": (s_sh, tp_sh, gp_sh, st_sh, res_sh),
        "separate": (s_sep, tp_sep, gp_sep, st_sep, res_sep),
        "comparison": cmp,
    }

## 使い方（1ファイルを試す）

このノートブックが `~/修士課程/BP/Models/` に置かれている想定で、`../statsbomb_data/` 以下のcsvを読み込む。

In [9]:
df = pd.read_csv("../statsbomb_data/Premier_League/PL2015-2016_goals_and_red_cards.csv")
state_occupancy(prepare_matches(df))

,state,side,exposure,goals,goals_per_match
0,00,away,154.463704,152,0.984050
1,lead_scored,away,72.059259,104,1.443257
2,lead_conceded,away,6.712778,9,1.340727
3,behind_scored,away,8.371667,14,1.672307
4,behind_conceded,away,101.641852,128,1.259324
5,level_scored,away,20.034074,27,1.347704
6,level_conceded,away,16.716667,25,1.495513
7,00,home,154.463704,196,1.268907
8,lead_scored,home,101.641852,178,1.751247
9,lead_conceded,home,8.371667,14,1.672307


In [10]:
out = fit_both(df, disp=True)

[stage1: L-BFGS-B] loglik=-555.5795 success=True
[stage2: Powell]   loglik=-555.5795 success=True
==== 直近得点者モデル（shared）====
試合数: 380, チーム数: 20, パラメータ数: 51
対数尤度: -555.579  AIC: 1213.159  BIC: 1414.108
収束: True (Optimization terminated successfully.)

---- 状態倍率 ----
          state  theta_home  theta_away
             00    1.000000    1.000000
    lead_scored    1.018899    1.018899
  lead_conceded    0.747765    0.747765
  behind_scored    1.126819    1.126819
behind_conceded    1.230399    1.230399
   level_scored    1.066236    1.066236
 level_conceded    0.806462    0.806462

---- 共通パラメータ ----
            parameter  estimate
              gamma_h  1.389702
                 rho1  4.467084
                 rho2  4.796265
    theta_lead_scored  1.018899
  theta_lead_conceded  0.747765
  theta_behind_scored  1.126819
theta_behind_conceded  1.230399
   theta_level_scored  1.066236
 theta_level_conceded  0.806462
                  xi1  0.516842
                  xi2  0.584118

[stage1: L-

## 全リーグ・全シーズンへの一括適用

共有・分離の両モデルを推定し、比較結果をまとめた一覧表と、状態倍率・チーム別パラメータのcsvを保存する。

In [11]:
DATA_DIR = "../statsbomb_data"
OUT_DIR = "./recent_scorer_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_goals_and_red_cards.csv"), recursive=True))
print(f"{len(csv_paths)} 件のcsvが見つかりました")

results_all = []
for path in csv_paths:
    print("=" * 60)
    print(path)
    df_i = pd.read_csv(path)
    league = df_i["competition_name"].iloc[0] if "competition_name" in df_i.columns and len(df_i) else os.path.basename(path)
    season = df_i["season_name"].iloc[0] if "season_name" in df_i.columns and len(df_i) else ""
    tag = f"{league}_{season}".replace("/", "-").replace(" ", "_")
    try:
        out_i = fit_both(df_i, disp=True, verbose=False)
        s_sh, s_sep, cmp = out_i["shared"][0], out_i["separate"][0], out_i["comparison"]
        results_all.append({
            "league": league, "season": season, "file": os.path.basename(path),
            "n_matches": s_sh["n_matches"],
            "loglik_shared": s_sh["log_likelihood"], "loglik_separate": s_sep["log_likelihood"],
            **cmp,
            "converged_shared": s_sh["converged"], "converged_separate": s_sep["converged"],
            "message_shared": s_sh["message"], "message_separate": s_sep["message"],
        })
        for name, (s, tp, gp, st, _) in [("shared", out_i["shared"]), ("separate", out_i["separate"])]:
            tp.to_csv(os.path.join(OUT_DIR, f"team_params_{name}_{tag}.csv"), index=False)
            gp.to_csv(os.path.join(OUT_DIR, f"global_params_{name}_{tag}.csv"), index=False)
            st.to_csv(os.path.join(OUT_DIR, f"state_table_{name}_{tag}.csv"), index=False)
        state_occupancy(prepare_matches(df_i)).to_csv(os.path.join(OUT_DIR, f"state_occupancy_{tag}.csv"), index=False)
    except Exception as e:
        print("失敗:", e)

summary_all_df = pd.DataFrame(results_all)
summary_all_df.to_csv(os.path.join(OUT_DIR, "summary_all.csv"), index=False)
summary_all_df

11 件のcsvが見つかりました
../statsbomb_data/FA_Women's_Super_League/WSL2018-2019_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-99.3314 success=False
[stage2: Powell]   loglik=-1000000105.1849 success=True
[stage1: L-BFGS-B] loglik=-99.1343 success=True
[stage2: Powell]   loglik=-1000000104.9679 success=True
../statsbomb_data/FA_Women's_Super_League/WSL2019-2020_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-61.9630 success=False
[stage2: Powell]   loglik=-60.4748 success=True
[stage1: L-BFGS-B] loglik=-1000000060.5213 success=False
[stage2: Powell]   loglik=-56.2017 success=True
../statsbomb_data/FA_Women's_Super_League/WSL2020-2021_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-90.9814 success=True
[stage2: Powell]   loglik=-90.9608 success=True
[stage1: L-BFGS-B] loglik=-1000000092.1602 success=False
[stage2: Powell]   loglik=-88.4323 success=True
../statsbomb_data/FA_Women's_Super_League/WSL2023-2024_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-82.6534 success=True
[stage2

,league,season,file,n_matches,loglik_shared,loglik_separate,LR_stat,df,p_value,AIC_shared,AIC_separate,BIC_shared,BIC_separate,converged_shared,converged_separate,message_shared,message_separate
0,FA Women's Super League,2018/2019,WSL2018-2019_goals_and_red_cards.csv,107,-1.000000e+09,-1.000000e+09,4.340541e-01,6,0.998551,2.000000e+09,2.000000e+09,2.000000e+09,2.000000e+09,True,True,[WARNING] 1 match(es) still have non-positive ...,[WARNING] 1 match(es) still have non-positive ...
1,FA Women's Super League,2019/2020,WSL2019-2020_goals_and_red_cards.csv,87,-6.047479e+01,-5.620166e+01,8.546271e+00,6,0.200749,1.909496e+02,1.944033e+02,2.772564e+02,2.955055e+02,True,True,Optimization terminated successfully.,Optimization terminated successfully.
2,FA Women's Super League,2020/2021,WSL2020-2021_goals_and_red_cards.csv,131,-9.096077e+01,-8.843226e+01,5.057028e+00,6,0.536520,2.519215e+02,2.588645e+02,3.525534e+02,3.767476e+02,True,True,Optimization terminated successfully.,Optimization terminated successfully.
3,FA Women's Super League,2023/2024,WSL2023-2024_goals_and_red_cards.csv,132,-8.265336e+01,-7.752234e+01,1.026205e+01,6,0.114042,2.353067e+02,2.370447e+02,3.362048e+02,3.552396e+02,True,True,Optimization terminated successfully.,Optimization terminated successfully.
4,Frauen Bundesliga,2023/2024,FB2023-2024_goals_and_red_cards.csv,132,-4.000000e+09,-1.179550e+02,8.000000e+09,6,0.000000,8.000000e+09,3.179100e+02,8.000000e+09,4.361048e+02,True,True,[WARNING] 4 match(es) still have non-positive ...,Optimization terminated successfully.
5,Indian Super league,2021/2022,ISL2021-2022_goals_and_red_cards.csv,115,-1.206773e+02,-1.204530e+02,4.486644e-01,6,0.998408,3.073546e+02,3.189059e+02,3.979374e+02,4.259583e+02,True,True,Optimization terminated successfully.,Optimization terminated successfully.
6,La Liga,2015/2016,LL2015-2016_goals_and_red_cards.csv,380,-5.211117e+02,-5.208042e+02,6.150442e-01,6,0.996144,1.144223e+03,1.155608e+03,1.345172e+03,1.380198e+03,True,True,Optimization terminated successfully.,Optimization terminated successfully.
7,Liga F,2023/2024,LF2023-2024_goals_and_red_cards.csv,240,-1.613735e+02,-1.549866e+02,1.277377e+01,6,0.046773,4.087469e+02,4.079732e+02,5.584144e+02,5.785245e+02,True,True,Optimization terminated successfully.,Optimization terminated successfully.
8,Ligue 1,2015/2016,L12015-2016_goals_and_red_cards.csv,377,-5.812509e+02,-5.801752e+02,2.151357e+00,6,0.905265,1.264502e+03,1.274350e+03,1.465046e+03,1.498488e+03,True,True,Optimization terminated successfully.,Optimization terminated successfully.
9,Premier League,2015/2016,PL2015-2016_goals_and_red_cards.csv,380,-5.555795e+02,-5.548049e+02,1.549202e+00,6,0.956159,1.213159e+03,1.223610e+03,1.414108e+03,1.448199e+03,True,True,Optimization terminated successfully.,Optimization terminated successfully.


## 注意点

- **データの薄い状態**：`lead_conceded` や `level_conceded` / `level_scored` は合計2点以上の試合でしか現れない。
  `state_occupancy` で滞在時間と得点数を確認し、得点数が少ない状態の倍率は解釈に注意する
  （境界 ±3（log）に張り付いていたら識別できていないサイン）。
- **元のモデルとの比較**：元のモデルVIとは入れ子関係にないため、尤度比検定ではなく AIC/BIC で比較する。
- **標準誤差**は出力していない。必要なら数値ヘシアンから近似できる。